# Homework 3 - Gradient Descent

In this homework, we will be using gradient descent algorithms and interpreting/visualizing the results of those algorithms.

In [ ]:
''' Setting up dependencies '''
import numpy as np  # General computations
import plotly.graph_objects as go  # Interactive plots
from typing import Callable as function  # Type annotations

##### HTML formatting helper for plotly graphs

In [ ]:
''' Setting up an HTML display helper function to help with formatting on later problems '''
# NOTE: Not currently used in this notebook due to weird HTML rendering behavior in my environment
import importlib.util, sys, subprocess

# Check whether IPython is available first; install only if missing
if importlib.util.find_spec("IPython") is None:
    print("IPython not found — installing ipython...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipython"])

from IPython.display import HTML, display

def details_block(title, inner_html, open: bool = False):
    ''' Defining some helper functions to help with formatting - 
    use plotly's built-in html exporting functionality to show output in dropdowns '''
    state = " open" if open else ""
    display(HTML(f"""
    <details{state} style="margin:8px 0">
      <summary><b>{title}</b></summary>
      <div style="margin:8px 0">{inner_html}</div>
    </details>
    """))

## Problem 1 - Initial Formula Creations

Defining a function for $f(x) = (x - 10)^2 + 100\text{sin}(\frac{x}{2} - \frac{\pi}{2})$ and it's derivative, $f'(x) = 2x - 20 + 50\text{sin}(\frac{x}{2})$

In [ ]:
def f(x):
    ''' Return the value of the function at x '''
    return ((x - 10) ** 2) + (100 * np.sin((x / 2) - (np.pi / 2)))
            
def fD(x):
    ''' Return the value of the derivative of the function at x '''
    return (2 * (x - 10)) + (50 * np.sin(x / 2))  # sin(x/2) equavalent to cos((x/2)-(pi/2))

In [ ]:
# Validating the functions
print(f"f(0) = {f(0)}, f(10) = {f(10)}")
print(f"fD(0) = {fD(0)}, fD(10) = {fD(10)}")

## Problem 2 - Visualizing the Cost Function

To do this, we can firstly define a function to create a 2d plot to plot the function as well as $(x_0, f(x_0))$ on the graph.

In [ ]:
def plot2DFunction(f, x0, x_min=-10, x_max=30, num_points=100, show_fig = False, theme = 'plotly_dark'):
    ''' Plot a single-variable function f and the point (x0, f(x0)) '''
    
    # Build the domains
    x = np.linspace(x_min, x_max, num_points)
    y = np.array([f(xi) for xi in x])
    
    # Build the figure
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='f(x)'))  # Add the function line
    fig.add_trace(go.Scatter(x=[x0], y=[f(x0)], mode='markers', name='x0'))  # Add the point x0
    fig.update_layout(
        title='f(x) with highlighted x0',  # Title for the plot 
        xaxis_title='x',  # X-axis label
        yaxis_title='f(x)',  # Y-axis label
        template=theme,  # Dark theme to look cool
    )
    
    if show_fig:
        fig.show()
    
    return fig

Now, we can use this function to visualize the $f(x)$ cost function at $x_0 = -4$.

In [ ]:
fx_plot = plot2DFunction(f, x0=-4, x_min=-10, x_max=10, show_fig=False)
fx_plot.show()

Additionally, we can expand out the range to see the function in a bigger picture.

In [ ]:
X_RANGE = (-50, 50)  # Modular config for the x range
fx_plot_wide = plot2DFunction(f, x0=-4, x_min=X_RANGE[0], x_max=X_RANGE[1], show_fig=False, theme='plotly_white')
fx_plot_wide.show()

## Problem 3 - 1D Gradient Descent Algorithm

Next, we can define a function named ```GradientDescent1D``` to help our later analysis.


In [ ]:
def GradientDescent1D(func: function, fprime: function, x0: float, alpha: float = 0.1, tol: float = 1e-5, max_iter: int = 1000, verbose : bool | int = 0):
    ''' 
    Performs a 1D gradient descent on a function.
    
    Parameters:
        func: A single-variable function f.
        fprime: The derivative of f.
        x0: Initial value x0.
        alpha: Optional learning rate.
        tol: Optional error tolerance for stopping.
        max_iter: Optional maximum number of iterations.
        verbose: Optional verbosity flag or integer to control print frequency.
    '''
    
    # Initialize current x, f(x), and f'(x)
    xk = x0
    fk = func(xk)
    fDk = fprime(xk)
     
    # Initialize steps and lists of coordinates for plotting
    num_iter = 0
    x_coords = [xk]
    y_coords = [fk]
    if verbose > 1: print(f"Iter {num_iter}: x = {xk}, f(x) = {fk}, f'(x) = {fDk}")
    
    # Begin the gradient descent loop - stop if max iterations reached or if the derivative is close to zero (within error tolerance)
    while num_iter <= max_iter and abs(fDk) > tol:
        # Update x using the gradient descent formula
        xk = xk - alpha * fDk
        
        # Recalculate f(x) and f'(x)
        fk = func(xk)
        fDk = fprime(xk)
        
        # Increment iteration count and store coordinates
        num_iter += 1
        x_coords.append(xk)
        y_coords.append(fk)
        
        # Print progress if verbosity is set (accepting numeric verbosity to control the frequency of prints)
        if (verbose or verbose > 0) and (isinstance(verbose, bool) or num_iter % verbose == 0 or abs(fDk) <= tol or num_iter == max_iter):
            print(f"Iter {num_iter}: x = {xk}, f(x) = {fk}, f'(x) = {fDk}")
    
    # Report results based on convergence
    if num_iter == max_iter:
            print("Warning: Maximum iterations reached without convergence.")
            if verbose or verbose > 0:
                print(f"Final: x = {xk}, f(x) = {fk}, f'(x) = {fDk}")
    else:
        print('Solution found:\n  y = {:.4f}\n  x = {:.4f}'.format(fk, xk))
        
    return (x_coords, y_coords)  # Return the list of coordinates as a tuple of lists

## Problem 4 - Testing the 1D Gradient Descent Algorithm

We can now test this function using the cost function from earlier ($\small{f(x) = (x - 10)^2 + 100\text{sin}(\frac{x}{2} - \frac{\pi}{2})}$), $x_0 = -4$, and $\alpha = 0.01$.

In [ ]:
# Defining the given parameters
X0 = -4
ALPHA = 0.01

# Calling the function with the given parameters, leaving the rest as defaults (storing the path as a tuple)
gd_path = GradientDescent1D(f, fD, x0=X0, alpha=ALPHA, verbose=15)

## Problem 5 - Visualizing the 1D Gradient Descent Algorithm

To visualize the results of the 1D Gradient Descent Algorithm, we can define a python function ```plotGDpath1d```, which will plot a given cost function, an initial value, and the gradient descent path as previously determined.

In [ ]:
def plotGDpath1d(f: function, init_val, gd_path: tuple = None, x_min = -30, x_max = 30, num_points = 1000, show_fig = False, theme: str = 'plotly_dark', **gd_kwargs):
    ''' 
    Plot the path taken by gradient descent on a single-variable function f 
    
    Parameters:
        f: A single-variable function f.
        init_val: Initial value x0.
        gd_path: Optional tuple of coordinate lists from the gradient descent algorithm. If None, will run the algorithm within the function.
        x_min: Optional minimum x-value for plotting.
        x_max: Optional maximum x-value for plotting.
        num_points: Optional number of points to plot the function.
        show_fig: Option flag to display the figure immediately.
        theme: Plotly theme for the figure.
        *gd_kwargs: Keyword argument list for the gradient descent algorithm, if no gd_path is provided.
    '''
    
    # Build the domains for f
    x = np.linspace(x_min, x_max, num_points)
    y = np.array([f(xi) for xi in x])
    
    # Unpack the gradient descent path if it exists
    if gd_path and isinstance(gd_path, tuple):
        gd_x_coords, gd_y_coords = gd_path
    # Otherwise, run the algorithm to get it
    else:
        gd_x_coords, gd_y_coords = GradientDescent1D(**gd_kwargs)
    
    # Build the figure
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='f(x)', line=dict(color='blue', width=2)))  # Add the function line
    fig.add_trace(go.Scatter(x=[init_val], y=[f(init_val)], mode='markers', name='Initial Value', marker=dict(color='yellow', size=8)))  # Clearly highlight the initial point
    fig.add_trace(go.Scatter(x=gd_x_coords, y=gd_y_coords, mode='markers+lines', name='GD Path', line=dict(color='orange', width=0.8), marker=dict(size=4)))  # Add the GD path
    fig.add_trace(go.Scatter(x=[gd_x_coords[-1]], y=[gd_y_coords[-1]], mode='markers', name='Final Point', marker=dict(color='red', size=8)))  # Clearly highlight the last point
    
    # Add titles and labels
    fig.update_layout(
        title='Gradient Descent Path on f(x)',  # Title for the plot 
        xaxis_title='x',  # X-axis label
        yaxis_title='f(x)',  # Y-axis label
        template=theme,
    )
    
    # Display if flag is set to true
    if show_fig: fig.show()
    
    return fig

Now, we can use to plot the current cost function, $x_0 = -20$, and $\alpha = 0.1$.

In [ ]:
X0 = -20  # Initial value
gd_kwargs = {"func": f, "fprime": fD, "x0": -20, "alpha": 0.1}  # Defining parameters for the gradient descent function (just learned this, very cool python functionality!)

# Calling the plot function
gd_fig = plotGDpath1d(f=f, init_val=X0, x_min=-20, x_max=5, **gd_kwargs)  # Adjusting x_min and x_max for better fit
gd_fig.show()

<!-- Describe results narratively -->
#### Interpreting the Results

When initially running the gradient descent algorithm on $x_0 = -20$ and $\alpha = 0.1$, the algorithm determines the local minimum to occur when $x = 2.9887$. However, when closely observing the plot, we can see that the algorithm fails to find the *true minimum*, reaching the lowest point in the path around the initial iterations. This can likely be attributed to the $\alpha$ parameter (learning rate) being set too high, with each iterative adjustment being too large and not allowing for the algorithm to approach the minimim.

To test this hypothesis, we can retest this with $\alpha = 0.01$. 

In [ ]:
gd_kwargs = {"func": f, "fprime": fD, "x0": -20, "alpha": 0.01}  # Decreasing alpha to 0.01
gd_fig = plotGDpath1d(f=f, init_val=X0, x_min=-20, x_max=5, **gd_kwargs)
gd_fig.show()

As we can see, this adjustment would cause the algorithm to move down the graph too slowly. To further test, we could adjust the initial value ($x_0$) to be -5 ($x_0 = -5$).

In [ ]:
gd_kwargs = {"func": f, "fprime": fD, "x0": -5, "alpha": 0.01}  # Adjusting alpha and x0
gd_fig = plotGDpath1d(f=f, init_val=-5, x_min=-5, x_max=5, **gd_kwargs)
gd_fig.show()

Now, we can observe that the algorithm reaches a lower local minimum.

### Designing an Algorithmic Approach to Finding the Global Minimum for this Cost Function

Given that we know the function $f(x) = (x - 10)^2 + 100\text{sin}(\frac{x}{2} - \frac{\pi}{2})$ and it's derivative $f'(x) = 2x - 20 + 50\text{sin}(\frac{x}{2})$, we can use mathematics to find the actual global minima. 

Firstly, we can observe an oscillating pattern with the $100\text{sin}(\frac{x}{2} - \frac{\pi}{2})$ portion of the function, since sin() oscillates between $[-1, 1]$, we can assume a reasonable range would be $[-100, 100]$. This is especially applicable as quadratic portion of the equation will dominate.

Given this, we can evaluate the y values at each $x$ in the range $[-100, 100]$ to further reduce the window.

In [ ]:
# Instantiate a list of y values and populate it with the function values at each x in the range [-50, 50]
x_vals = list(range(-100, 101))
y_vals = [f(xi) for xi in x_vals]

# Now, we can find the minimum whole-number values to further reduce our search window
min_y = min(y_vals)
min_y_index = y_vals.index(min_y)
min_x = x_vals[min_y_index]
optimal_window = (min_x - 3, min_x + 3)  # Window of 10 around the minimum x value
print(f"Optimal window for x: {optimal_window}, with minimum y: {min_y} at x = {min_x}")

Now that we have a more precise window, we can run the gradient descent algorithm with refined parameters:
- $x_0 = 12$
- $\alpha = .001$
- $\text{tol} = 1e-9$

In [ ]:
# Defining the new parameter set for the gradient descent algorithm
gd_params_optimized = {
    "func": f, 
    "fprime": fD, 
    "x0": 12,  # Starting near the observed whole-number minimum
    "alpha": 0.001, # Using a low learning rate, since we are already close to the minimum
    "tol": 1e-9,  # Very low tolerance for high precision
    "verbose": 50,  # Print every 50 iterations
    "max_iter": 10000,  # Allowing more iterations for convergence
}

# Running the plot function with the optimized parameters
gd_fig_optimized = plotGDpath1d(f=f, init_val=gd_params_optimized["x0"], x_min=optimal_window[0], x_max=optimal_window[1], **gd_params_optimized)
gd_fig_optimized.show()

For even more over-kill precision, we could also run it again given the new perceived global minimum of $x = 12.3760$ with an even lower error tolerance.

In [ ]:
# Defining the new parameter set for the gradient descent algorithm
gd_params_super_optimized = {
    "func": f, 
    "fprime": fD, 
    "x0": 12.3760,  # Starting near the previously observed minimum
    "alpha": 0.00001, # Decreasing alpha even more for finer adjustments
    "tol": 1e-12,  # Even lower tolerance for higher precision
    "verbose": 50000,  # Print every 50 iterations
    "max_iter": 100000,  # Increasing max iterations even more for convergence
}

# Running the plot function with the optimized parameters
gd_fig_super_optimized = plotGDpath1d(f=f, init_val=gd_params_super_optimized["x0"], x_min=12.35, x_max=12.395, **gd_params_super_optimized)
gd_fig_super_optimized.show()

Now, we can see that the new perceived global minima occurs when $x = 12.376003047239983$. 

## Problem 6 - Higher Dimensional Plots

Next, we are given the *Himmelblau Function*:
$$H(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2$$

In [ ]:
# Defining the Himmelblau Function
def himmelblau(x, y):
    ''' Return the value of the Himmelblau function '''
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

To visualize this, we can define and run a function that generates a 3D plot.

In [ ]:
def plot3DFunction(f, x_min=-10, x_max=10, y_min=-10, y_max=10, num_points=1000, show_fig = False, theme = 'plotly_dark'):
    ''' 
    Plot a two-variable function f(x, y) in 3D.
    
    Parameters:
        f: A two-variable function f(x, y).
        x_min: Optional minimum x-value for plotting.
        x_max: Optional maximum x-value for plotting.
        y_min: Optional minimum y-value for plotting.
        y_max: Optional maximum y-value for plotting.
        num_points: Optional number of points along each axis for plotting.
        show_fig: Optional flag to display the figure immediately.
        theme: Optional plotly theme for the figure.
    '''
    
    # Build the meshgrid for f
    x = np.linspace(x_min, x_max, num_points)
    y = np.linspace(y_min, y_max, num_points)
    X, Y = np.meshgrid(x, y)
    Z = np.array([[f(xi, yi) for xi in x] for yi in y])
    
    # Build the figure
    fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y)])
    
    # Add titles and labels
    fig.update_layout(
        title='3D Surface Plot of f(x, y)',  # Title for the plot 
        scene=dict(
            xaxis_title='x',  # X-axis label
            yaxis_title='y',  # Y-axis label
            zaxis_title='f(x, y)'  # Z-axis label
        ),
        template=theme,
    )
    
    # Display if flag is set to true
    if show_fig: fig.show()
    
    return fig

temp = plot3DFunction(himmelblau, show_fig=True)

## Problem 7 - Gradient Calculation

In order to calculate the gradient for $H(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2$, we must firstly find the partial derivatives in respect to $x$ and $y$. When solving these, we get:
<!-- Had to mess with formatting to make my markdown renderer happy -->
$$ 
\frac{\partial f}{\partial x} = 4x(x^2 + y - 11) + 2(x + y^2 - 7)
$$

$$ 
\frac{\partial f}{\partial y} = 2(x^2 + y - 11) + 4y(x + y^2 - 7)
$$

We can then place these in vector form to get the gradient, giving us:
$$
\nabla f(x,y) = 
\begin{bmatrix}  % Syntax for making the vector thing
4x(x^2 + y - 11) + 2(x + y^2 - 7)\\
2(x^2 + y - 11) + 4y(x + y^2 - 7)
\end{bmatrix}
= 
\begin{bmatrix}
4x^{3} + 4xy + 2y^{2} - 42x - 14\\
2x^{2} + 4xy + 4y^{3} - 26y - 22
\end{bmatrix}
$$

In [ ]:
# Making python definitions for the Himmelblau function and its gradient
def gradientHimmelblau(x: float = None, y: float = None):
    """
    Return partial derivatives (df/dx, df/dy) for the Himmelblau function
    """
    dfdx = 4 * x * (x**2 + y - 11) + 2 * (x + y**2 - 7)
    dfdy = 2 * (x**2 + y - 11) + 4 * y * (x + y**2 - 7)
    return dfdx, dfdy

## Problem 8 - Gradient Descent in High Dimensions

Now, we will implement a 2D Gradient Descent algorithm that returns the $x, y,$ and $z$ coordiantes of the Gradient Descent path.

In [ ]:
def GradientDescent2D(f, gradientF, x0: tuple, alpha = 0.01, tol = 1e-5, max_iter = 1000, verbose : bool | int = 0):
    ''' 
    Performs a 2D gradient descent on a function.
    
    Parameters:
        func: A two-variable function f(x, y).
        gradientF: The gradient function of f (the partial derivatives (df/dx, df/dy)).
        x0: Initial value as a tuple (x0, y0).
        alpha: Learning rate.
        tol: Error tolerance for stopping.
        max_iter: Maximum number of iterations.
        verbose: Optional verbosity flag or integer to control print frequency.
    '''
    
    # Initialize the current x values (xk), f(xk), and the gradient at xk
    xk = np.array(x0, dtype=float)
    fk = f(*xk)  # Unpack the x, y tuple
    gfk = np.array(gradientF(*xk), dtype=float)  # Return as a numpy array
    gfk_norm = np.linalg.norm(gfk)
    
    # Initializing the number of steps and lists of coordinates for plotting
    num_iter = 0
    xy_coords = [ xk ]
    z_coords = [ fk ]
    if verbose > 1: print(f"Iter {num_iter}: x = {xk}, f(x) = {fk}, f gradient = {gfk}")
    
    # Loop until the maximum number of iterations is reached or the gradient norm is within the tolerance
    while num_iter <= max_iter and gfk_norm > tol:
        # Update x variables using the gradient descent formula
        xk = xk - alpha * gfk
        
        # Recalculate f(x)
        fk = f(*xk)
        
        # Break early if f(x) becomes NaN or infinite
        if np.isnan(fk) or np.isinf(fk):
            print("Warning: f(x) became NaN or infinite. Stopping early.")
            break
        
        # Recalculate the gradient at xk
        gfk = np.array(gradientF(*xk), dtype=float)  # Force compatible types
        gfk_norm = np.linalg.norm(gfk)
        
        # Increment counter and record the latest coordinates
        num_iter += 1
        xy_coords.append(xk)
        z_coords.append(fk)
        
        # Print progress if verbosity is set (accepting numeric verbosity to control the frequency of prints)
        if (verbose or verbose > 0) and (isinstance(verbose, bool) or num_iter % verbose == 0 or num_iter == max_iter):
            print(f"Iter {num_iter}: x, y = {xk}, f(x) = {fk}, f gradient = {gfk}")
    
        
    # Printing the results to the console (only if verbose is set)
    if num_iter == max_iter and verbose:
        print("Warning: Maximum iterations reached without convergence.")
        print(f"Final: z = {fk}, x, y = ({xk[0]}, {xk[1]})")
    elif verbose:
        print('Solution found:\n  z = {:.4f}\n  x, y = ({:.4f}, {:.4f})'.format(fk, xk[0], xk[1]))
        
    # Return the list of coordinates
    return ( xy_coords, z_coords )

## Problem 9 - Countour Plots

Firstly, we can define a function make a Gradient Descent Countour Plot.

In [ ]:
def plotGDCountour2D(f, gradientFPath = None, x_min = -10, x_max = 10, y_min = -10, y_max = 10, \
    num_points = 500, show_fig: bool = False, theme: str = 'plotly_dark', title: str = 'Gradient Descent Path on f(x, y)',**gd_kwargs):
    ''' 
    Plot the contour of a two-variable function f(x, y) and the path taken by gradient descent.
    
    Parameters:
        f: A two-variable function f(x, y).
        x0: Initial value as a tuple (x0, y0).
        gradientFPath: Optional path of the gradient function of f. If none is provided, the function will run the gradient descent algorithm internally.
        alpha: Optional learning rate.
        tol: Optional error tolerance for stopping.
        max_iter: Optional maximum number of iterations.
        x_min: Optional minimum x-value for plotting.
        x_max: Optional maximum x-value for plotting.
        y_min: Optional minimum y-value for plotting.
        y_max: Optional maximum y-value for plotting.
        show_fig: Optional flag to display the figure immediately.
        theme: Optional plotly theme for the figure.
        **gd_kwargs: Keyword argument list for the gradient descent algorithm, if no gd_path is provided.
    '''
    
    # Unpack the gradient descent path if it exists
    if gradientFPath and isinstance(gradientFPath, tuple):
        gd_xy_coords, gd_z_coords = gradientFPath
    # Otherwise, run the algorithm to get it
    else:
        gd_xy_coords, gd_z_coords = GradientDescent2D(**gd_kwargs)
    
    # Build the meshgrid for f
    x = np.linspace(x_min, x_max, num_points)
    y = np.linspace(y_min, y_max, num_points)
    Z = np.array([[f(xi, yi) for xi in x] for yi in y])
    
    # Build the contour figure
    fig = go.Figure(data=go.Contour(
        z=Z, x=x, y=y,
        colorscale='Plasma',  # Cool colorscale
        ncontours=50,  # Adding more contour lines for more describtive visualization
        contours=dict(coloring='heatmap', showlabels=True, labelfont=dict(color='white')),
        colorbar=dict(title='f(x, y)')
    ))
    
    # Add the gradient descent path
    gd_xy_coords = np.array(gd_xy_coords, dtype=float)
    fig.add_trace(go.Scatter(x=gd_xy_coords[:, 0], y=gd_xy_coords[:, 1], mode='markers+lines', name='GD Path', line=dict(color='orange', width=2), marker=dict(size=6)))
    
    # Mark the start and end points
    fig.add_trace(go.Scatter(x=[gd_xy_coords[0, 0]], y=[gd_xy_coords[0, 1]], mode='markers', name='Start', marker=dict(color='yellow', size=10)))
    fig.add_trace(go.Scatter(x=[gd_xy_coords[-1, 0]], y=[gd_xy_coords[-1, 1]], mode='markers', name='End', marker=dict(color='red', size=10)))
    
    # Update the formatting
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", yanchor="top", font=dict(size=24)),  # Centered title for the plot 
        xaxis_title='x',  # X-axis label
        yaxis_title='y',  # Y-axis label
        template=theme,
        legend=dict(orientation="h", yanchor="top", y=1.15, xanchor="right", x=1.02),  # Horizontal legend at the top
        margin=dict(r=90)  # Tight margins
    )
    
    if show_fig: fig.show()  # Display if flag is set to true
    

    return fig

Now that we defined the function, we can test the following initial value vectors:
- $(0, 3)$
- $(0, 0)$
- $(0, -5)$
- $(0.2, -5)$

Also testing the possibilities where $\alpha = 0.1, 0.01,$ and $0.02$.

In [ ]:
# Defining the parameters we would like to test
initial_points = [(0, 3), (0, 0), (0, -5), (0.2, -5)]  # List of tuples of initial points
alphas = [0.01, 0.02, 0.1]  # List of learning rates to test

# Running the gradient descent and plotting for each combination of initial point and alpha
for alpha in alphas:
    for point in initial_points:
        print(f"\nTesting initial point {point} with alpha = {alpha}")
        gd_path = GradientDescent2D(f=himmelblau, gradientF=gradientHimmelblau, x0=point, alpha=alpha, tol=1e-5, max_iter=1000, verbose=1)  # Get the gradient descent path to pass into plotting function
        contour_fig = plotGDCountour2D(himmelblau, x0=point, gradientFPath=gd_path, x_min=-6, x_max=6, y_min=-6, y_max=6, num_points=200, \
            theme='plotly_dark', title=f'GD Path from {point} with alpha={alpha}')  # Plot the contour with the path
        contour_fig.show()  # Seeing the plots directly
        # Display the plot in a dropdown for cleaner output - Commented out due to weird HTML rendering behavior in some environments
        # html = contour_fig.to_html(full_html=False, include_plotlyjs='inline')
        # details_block(f'Plot for x0={point}, alpha={alpha}', html)

<!-- Describe results narratively -->
### Interpreting the Results

Based on the results of the contour plots, we can derive conclusions from each set of $x_0$ and $\alpha$.

<!-- NOTE TODO ONCE DONE INTERPRETING ALL OF THE RESULTS-->
<!-- **Summary of the results**: The strongest start parameters includes $\alpha = 0.01$, which clearly converges to a local minimum. The $x_0$ parameter of $(0, 3)$ and $(0, -5)$ were also both good starting points, each quickly converging with minimal iterations. However, $(0, 5)$ converged slightly faster, taking a deeper path to a local minimum of $z = 0$. -->

#### $x_0 = (0, 3), \alpha = 0.01$

This successfully converges to $z=0$ at iteration 28, showing a clear and fast convergence. This is also clear in the contour plot, suggesting a strong starting $x_0$ and $\alpha$

#### $x_0 = (0, 0), \alpha = 0.01$

This takes 54 iterations to converge to $z=0$, nearly 20 more than the previous set of parameters. The contour plot also suggests that the path taken was less steep and that $x_0$ was a higher starting point than the previous parameter, which would account for the increased number of iterations.

#### $x_0 = (0, -5), \alpha = 0.01$

This only took 25 iterations to converge to $z=0$, suggesting superior performance. When observing the countour plot, the $x_0$ parameter began at a significantly higher $z$ value, however took a much steeper descent to reach a local minimum.

#### $x_0 = (0.2, -5), \alpha = 0.01$

This successfully converged to $z=0$ after 57 iterations, taking the longest out of all $x_0$ parameters for $\alpha=0.01$. The contour plot highlights that, despite a steep descent after starting at a high $z$ value, it took many iterations to acheive the minimum near the end.

#### $x_0 = (0, 3), \alpha = 0.02$

This parameter set successfully converged to $z=0$ in only 21 iterations, being the most promising parameter set yet. The contour plot highlights how it took a straight and somewhat steep path, starting off with lower $(x, y)$ steps but quickly taking larger steps torwards the local minimum. This can be attested to the higher learning rate.

#### $x_0 = (0, 0), \alpha = 0.02$

This parameter was similar to it's $\alpha = 0.01$ counterpart, however it was able to converge to $z=0$ over 20 iterations faster. It took a very steep path until it could successfully reach it's local minimum, suggesting that the higher $\alpha$ may be superior in this scenario.

#### $x_0 = (0, -5), \alpha = 0.02$

This parameter set took 29 iterations, taking an extremely steep initial path while mellowing out to very small changes at the end. It was able to converge to $z=0$ in a few more iterations than it's $\alpha = 0.01$ counterpart, suggesting that the higher $\alpha$ caused the path to overstep and need to make numerous corrections.

#### $x_0 = (0.2, -5), \alpha = 0.02$

This parameter set took only 25 iterations to converge to $z=0$, nearly half as much of that when $\alpha = 0.01$. It started at an extremely high $z$ value at $x_0$, however took very steep steps torwards the local minimum.

#### $\alpha = 0.1$

For all scenarios where $\alpha = 0.1$, the gradient descent algorithm was unable to complete due to f(x) quickly approaching infinity. This is likely due 


## Problem 10 $\text{\tiny (Challenge)}$ - 3D-Plot with GD Path
Using plotly, we can create a 3D plot for the results of from the previous trial.

In [ ]:
def plotGDPath3D(f, gradientFPath = None, x_min = -10, x_max = 10, y_min = -10, y_max = 10, show_fig: bool = False, theme: str = 'plotly_dark', num_points = 100, **gd_kwargs):
    ''' 
    Plot the 3D surface of a two-variable function f(x, y) and the path taken by gradient descent.
    
    Parameters:
        f: A two-variable function f(x, y).
        gradientFPath: Optional path of the gradient function of f. If none is provided, the function will run the gradient descent algorithm internally.
        x_min: Optional minimum x-value for plotting.
        x_max: Optional maximum x-value for plotting.
        y_min: Optional minimum y-value for plotting.
        y_max: Optional maximum y-value for plotting.
        show_fig: Optional flag to display the figure immediately.
        theme: Optional plotly theme for the figure.
        **gd_kwargs: Keyword argument list for the gradient descent algorithm, if no gd_path is provided.
    '''
    
    # Unpack the gradient descent path if it exists
    if gradientFPath and isinstance(gradientFPath, tuple):
        gd_xy_coords, gd_z_coords = gradientFPath
    # Otherwise, run the algorithm to get it
    else:
        if 'f' not in gd_kwargs:  # Set f default due to overlapping parameter names
            gd_kwargs['f'] = f
        gd_xy_coords, gd_z_coords = GradientDescent2D(**gd_kwargs)
        
    # Convert to numpy arrays for slicing
    gd_xy_coords = np.array(gd_xy_coords, dtype=float)
    gd_z_coords = np.array(gd_z_coords, dtype=float)
    
    # Build the meshgrid for f
    x = np.linspace(x_min, x_max, num_points)
    y = np.linspace(y_min, y_max, num_points)
    Z = np.array([[f(xi, yi) for xi in x] for yi in y])
    
    # Build the figure
    fig = go.Figure(data=[go.Surface(z=Z, x=x, y=y, colorscale='Plasma', opacity=0.8)])  # Adding the actual 3d shape
    # Add the GD path
    fig.add_trace(go.Scatter3d(x=gd_xy_coords[:, 0], y=gd_xy_coords[:, 1], z=gd_z_coords, mode='markers+lines', name='GD Path', line=dict(color='orange', width=2), marker=dict(size=2)))
    # Add the start and end points
    fig.add_trace(go.Scatter3d(x=[gd_xy_coords[0, 0]], y=[gd_xy_coords[0, 1]], z=[gd_z_coords[0]], mode='markers', name='Start', marker=dict(color='yellow', size=4)))  # Start point
    fig.add_trace(go.Scatter3d(x=[gd_xy_coords[-1, 0]], y=[gd_xy_coords[-1, 1]], z=[gd_z_coords[-1]], mode='markers', name='End', marker=dict(color='red', size=4)))  # End point
    
    # Update the formatting
    fig.update_layout(
        title=dict(text='3D Gradient Descent Path on f(x, y)', x=0.5, xanchor="center", yanchor="top", font=dict(size=24)),  # Centered title for the plot 
        scene=dict(
            xaxis_title='x',  # X-axis label
            yaxis_title='y',  # Y-axis label
            zaxis_title='f(x, y)'  # Z-axis label
        ),
        template=theme,
        legend=dict(orientation="h", yanchor="top", y=1.15, xanchor="right", x=1.02),  # Horizontal legend at the top
        margin=dict(r=90)  # Tight margins
    )
    
    if show_fig: fig.show()  # Display if flag is set to true
    
    return fig

Now, we can test this function using the initial value vector of $(0.2, -5)$ and $\alpha =  0.01$.

In [ ]:
gd_args = {"gradientF": gradientHimmelblau, "x0": (0.2, -5), "alpha": 0.01, "tol": 1e-5, "max_iter": 1000, "verbose": False}  # Parameters for the gradient descent function
# Plot using the above function
gd_3d_fig = plotGDPath3D(f=himmelblau, x_min=-6, x_max=6, y_min=-6, y_max=6, num_points=200, show_fig=False, theme='plotly_dark', **gd_args)
gd_3d_fig.show()

## Problem 11 - References
- [Plotly's Beautiful Documentation](https://plotly.com/python/)
- [IPython (Interactive Python) Documentation (display HTML)](https://ipython.readthedocs.io/en/9.0.2/api/generated/IPython.display.html)
- [Python Tips (**kwargs)](https://book.pythontips.com/en/latest/args_and_kwargs.html)